In [ ]:
#export 

import logging
import fitz
import re
import io
from typing import Tuple
from PIL import Image
import unicodedata
from docx import Document
from mlx_vlm import load, generate
from mlx_vlm.prompt_utils import apply_chat_template

import sys
from pathlib import Path

class PDFTranscriber:
    _class_name = "PDFTranscriber"

    DEFAULT_PROMPT = """
    Transcribe esta página de forma literal y completa.
    
    #Instrucciones:
    - Devuelve únicamente la transcripción.
    - Mantén el orden de lectura natural, empezando de izquierda a derecha y de arriba a abajo.
    - Conserva títulos, párrafos, listas, tablas simples y saltos de línea razonables.
    - No resumas.
    - No expliques nada.
    - No corrijas el contenido.
    - No añadas texto que no esté en la imagen.
    - Si hay texto dudoso, escribe la mejor lectura posible.
    - Si una zona es ilegible, marca solo esa zona como [ilegible].
    """

    DEFAULT_MODEL = "mlx-community/Qwen3.5-35B-A3B-4bit"
    OLD_MODEL = "mlx-community/Qwen3.5-9B-MLX-4bit"
    GEMMA_MODEL = ""

    def __init__(
        self,
        pdf_path:str,
        output_dir: Path,
        dpi: int = 200,
        max_tokens: int = 8192,
        temperature: float = 0.0,
        prompt: str = None,
        overwrite: bool = False,
        model_name: str = None,
        log_level: int = logging.INFO
    ) -> None:
        method="__init__"

        self.model_name = model_name or self.DEFAULT_MODEL
        self.output_dir = Path(output_dir)
        self.pdf_path = Path(pdf_path)
        self.dpi = dpi
        self.max_tokens = max_tokens
        self.temperature = temperature
        self.prompt = prompt or self.DEFAULT_PROMPT
        self.overwrite = overwrite

        self.logger = self._create_logger(log_level)
        self.logger.info(f"{self._class_name}.{method}-START Cargando modelo: {self.model_name}, con path: {output_dir} y fichero: {pdf_path}")

        if not pdf_path.exists():
            raise FileNotFoundError(f"{self._class_name}.{method}- No existe el PDF: {pdf_path}")
        
        if not self.output_dir.exists():
            self.output_dir.mkdir(parents=True, exist_ok=True)

        self.logger.info(f"{self._class_name}.{method}- cargando el modelo {self.model_name}")
        self.model, self.processor = load(self.model_name)

    ######################################
    ## Métodos privados de la clase
    ######################################

    def _create_logger(self, log_level: int) -> logging.Logger:
        logger = logging.getLogger(self.__class__.__name__)

        if not logger.handlers:
            handler = logging.StreamHandler()
            formatter = logging.Formatter(
                fmt="%(asctime)s %(levelname)s [%(name)s] %(message)s",
                datefmt="%Y-%m-%d %H:%M:%S",
            )
            handler.setFormatter(formatter)
            logger.addHandler(handler)

        logger.setLevel(log_level)
        logger.propagate = False
        return logger
    
    def _already_done(self, out_file: Path, overwrite: bool) -> bool:
        return out_file.exists() and not overwrite
    
    def _render_page_to_pil(self, page: fitz.Page, dpi: int) -> Image.Image:
        """
        Renderiza una página PDF a imagen PIL en RGB.
        """
        zoom = dpi / 72.0
        matrix = fitz.Matrix(zoom, zoom)
        pix = page.get_pixmap(matrix=matrix, alpha=False)
        img_bytes = pix.tobytes("png")
        image = Image.open(io.BytesIO(img_bytes)).convert("RGB")
        return image
    
    def _save_page_image(self, image: Image.Image, output_dir: Path) -> Path:
        """
        Guarda la imagen renderizada de una página en el directorio de salida.
        """
        image_path = output_dir
        image.save(image_path, format="PNG")
        return image_path

    def _save_text(self, text: str, path: Path) -> None:
        path.write_text(text, encoding="utf-8")
    
    def _split_thinking_and_answer(self, text: str) -> Tuple[str, str]:
        if not text:
            return "", ""

        text = text.strip()
        #pattern = re.compile(r"<think>\s*(.*?)\s*</think>\s*(.*)", re.DOTALL)
        #match = pattern.fullmatch(text)

        #if match:
        #    return match.group(1).strip(), match.group(2).strip()

        #return "", text
        if "</think>" not in text:
            return "", text

        #before, after = text.split("</think>", 1)

        #thinking = re.sub(r"^\s*<think>\s*", "", before).strip()
        #answer = after.strip()

        #return thinking, answer
        pattern = re.compile(r"(?:<think>\s*)?(.*?)\s*</think>\s*(.*)", re.DOTALL)
        match = pattern.fullmatch(text)

        if match:
            return match.group(1).strip(), match.group(2).strip()
    
    def _build_prompt(self, processor, model, base_prompt: str) -> str:
        """
        Aplica el chat template esperado por mlx-vlm para 1 imagen.
        """
        config = getattr(model, "config", None)
        if config is None:
            raise RuntimeError("No se pudo obtener model.config del modelo cargado.")

        formatted = apply_chat_template(
            processor,
            config,
            base_prompt,
            num_images=1,
        )
        return formatted

    def _transcribe_page(self, image):
        formatted_prompt = self._build_prompt(self.processor, self.model, self.prompt)

        output = generate(
            self.model,
            self.processor,
            formatted_prompt,
            [image],
            max_tokens=self.max_tokens,
            temperature=self.temperature,
            verbose=False,
            # si tu versión lo soporta:
            thinking_budget=64,
            thinking_start_token="<think>",
            thinking_end_token="</think>",
        )

        raw_text = "" if output is None else str(output.text).strip()
        thinking, answer = self._split_thinking_and_answer(raw_text)

        return thinking, answer

    ######################################
    ## Métodos públicos de la clase
    ######################################
    
    def process_pdf(self,
        start_page: int = None,
        end_page: int = None,
    ) -> None:
        method="process_pdf"

        self.logger.info(f"{self._class_name}.{method}-START procesando fichero {self.pdf_path}")

        #Abrimos el fichero y calculamos las páginas que vamos a procesar. 
        doc = fitz.open(self.pdf_path)
        total_pages = len(doc)
        width=len(str(total_pages))
        first = 1 if start_page is None else max(1, start_page)
        last = total_pages if end_page is None else min(total_pages, end_page)
        self.logger.info(f"{self._class_name}.{method}- páginas Inizio={first}, Final={last} de {total_pages}.")

        if first > last:
            raise ValueError(
                f"Rango de páginas inválido: start_page={first}, end_page={last}, total={total_pages}"
            )

        for page_number in range(first, last + 1):
            out_file_txt = self.output_dir / f"pagina_{page_number:0{width}d}.txt"
            out_file_error_txt = self.output_dir / f"pagina_{page_number:0{width}d}_error.txt"
            out_file_txt_thinking = self.output_dir / f"pagina_{page_number:0{width}d}.dbg"
            out_file_png = self.output_dir / f"pagina_{page_number:0{width}d}.png"

            #Preparamos el fichero de imagen
            try:
                page = doc.load_page(page_number - 1)
                image = self._render_page_to_pil(page, dpi=self.dpi)

                if self._already_done(out_file_png, self.overwrite):
                    self.logger.info(f"{self._class_name}.{method}- [SKIP] Ya existe: {out_file_png}")
                else:
                    self._save_page_image(image, out_file_png)
            except Exception as e:
                error_file = self.output_dir / f"pagina_{page_number}_error_png.txt"
                msg = f"Error en página {page_number}: {e}\n"
                self._save_text(msg, error_file)
                self.logger.error(f"{self._class_name}.{method}-ERROR {msg}")
                    
            #Preparamos el fichero de texto con el LLM.
            if self._already_done(out_file_txt, self.overwrite):
                self.logger.info(f"{self._class_name}.{method}- [SKIP] Ya existe: {out_file_txt}")
                continue
            
            #Detectamos en la fase de generación si hay una respuesta anómala y se vuelve a generar.
            error = True
            max_retries = 3
            retries = 0
            
            while error and retries < max_retries:
                self.logger.info(f"{self._class_name}.{method}- Transcribiendo página {page_number}...")
                thinking, text = self._transcribe_page(image)
                text_clean = text.lstrip() if isinstance(text, str) else ""
                retry_needed = (thinking in (None, "")
                    or text_clean.startswith("The user")
                    or text_clean.startswith("<thinking>"))
                
                if not retry_needed:
                    self._save_text(text_clean, out_file_txt)
                    error = False
                    #self._save_text(thinking, out_file_txt_thinking)
                else:
                    retries += 1
                    self.logger.warning(f"{self._class_name}.{method}- Respuesta inválida en página {page_number}. Reintento {retries}/{max_retries}. Thinking={thinking}, Text={text_clean}")
        
            if error:
                self.logger.error(f"{self._class_name}.{method}- No se pudo transcribir correctamente la página {page_number}")
                text_clean="ERROR transcribiendo la página."
                self._save_text(text_clean, out_file_error_txt)
  
        doc.close()
        self.logger.info(f"{self._class_name}.{method}- Proceso finalizado")
    
    def genera_docx(self) -> Path:
        """
        Recorre un directorio, recoge todos los .txt ordenados por nombre
        y los combina en un .docx con el mismo nombre que el directorio.

        Ejemplo:
        /ruta/salida/mi_pdf/
            pagina_0001.txt
            pagina_0002.txt

        genera:
        /ruta/salida/mi_pdf/mi_pdf.docx
        """
        method="genera_docx"
        output_dir = self.output_dir

        txt_files = sorted(output_dir.glob("*.txt"), key=lambda p: p.name)

        if not txt_files:
            raise FileNotFoundError(f"{self._class_name}.{method}-ERROR No se encontraron ficheros .txt en: {output_dir}")

        #Creamos un documento nuevo.
        doc = Document()
        doc.add_heading(output_dir.name, level=0)

        for i, txt_file in enumerate(txt_files):
            content = txt_file.read_text(encoding="utf-8").strip()
            doc.add_heading(txt_file.stem, level=1)

            # Añade el contenido del txt
            if content:
                for line in content.splitlines():
                    doc.add_paragraph(line)
            else:
                doc.add_paragraph("")

            # Salto de página entre ficheros, salvo en el último
            #if i < len(txt_files) - 1:
            #    doc.add_page_break()

        docx_path = output_dir / f"{output_dir.name}.docx"
        doc.save(docx_path)

        return docx_path

/Users/zzddfge/.pyenv/versions/mlx-31211/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
import time 

usuario = "Users/zzddfge"
carpeta = "Downloads"
disco_duro = "/Volumes/Macintosh HD"
fichero_pdf = "Política Criminal Libro nuevo.pdf"

def nfc(s: str) -> str:
    return unicodedata.normalize("NFC", s)

output_path= Path(nfc(disco_duro)) / nfc(usuario) / nfc(carpeta) / nfc("Política Criminal")
pdf_path= Path(nfc(disco_duro)) / nfc(usuario) / nfc(carpeta) / nfc(fichero_pdf)

pdf = PDFTranscriber(pdf_path=pdf_path, output_dir=output_path)

#Capítulo 1 y 2
pdf.process_pdf(6, 43)

#Capítulo 3 y 4
pdf.process_pdf(44, 76)
pdf.process_pdf(77, 96)

#Capítulo 5 y 6
pdf.process_pdf(97, 118)
pdf.process_pdf(119, 144)

#Capítulo 7 y 8
pdf.process_pdf(145, 172)
pdf.process_pdf(173, 188)

#Capítulo 9 y 10
pdf.process_pdf(189, 210)
pdf.process_pdf(211, 224)

#Capítulo 11
pdf.process_pdf(225, 248)

pdf.genera_docx()

In [ ]:
import time 

usuario = "Users/zzddfge"
carpeta = "Downloads"
disco_duro = "/Volumes/Macintosh HD"
fichero_pdf = "Economía y Estrategia Medioambiental Libro.pdf"

def nfc(s: str) -> str:
    return unicodedata.normalize("NFC", s)

output_path= Path(nfc(disco_duro)) / nfc(usuario) / nfc(carpeta) / nfc("Economia y Estrategia Medioambiental")
pdf_path= Path(nfc(disco_duro)) / nfc(usuario) / nfc(carpeta) / nfc(fichero_pdf)

pdf = PDFTranscriber(pdf_path=pdf_path, output_dir=output_path)

#Capítulo 1 y 2
pdf.process_pdf(30, 47)

#Capítulo 3 y 4
pdf.process_pdf(48, 73)

#Capítulo 5 y 6
pdf.process_pdf(74, 110)

#Capítulo 7 y 8
pdf.process_pdf(111, 153)

#Capítulo 9 y 10
pdf.process_pdf(154, 196)

pdf.genera_docx()

In [ ]:
from ollama import chat

prompt="""
    Transcribe esta página de forma literal y completa.
    
    #Instrucciones:
    - Devuelve únicamente la transcripción.
    - Mantén el orden de lectura natural, empezando de izquierda a derecha y de arriba a abajo.
    - Conserva títulos, párrafos, listas, tablas simples y saltos de línea razonables.
    - No resumas.
    - No expliques nada.
    - No corrijas el contenido.
    - No añadas texto que no esté en la imagen.
    - Si hay texto dudoso, escribe la mejor lectura posible.
    - Si una zona es ilegible, marca solo esa zona como [ilegible].
    """

output_path_file= output_path / "pagina_9.png"
response = chat(
    model='qwen3.5:35b',
    messages=[
        {
            'role': 'user',
            'content': prompt,
            'images': [output_path_file],
        }
    ]
)

print(response['message']['content'])

OBJETIVOS DE LA LECCIÓN

En este primer capítulo se trata de acercarse al concepto de Política Criminal y de tener en cuenta los distintos sentidos en que nos podemos referir a la misma para comprender qué se entiende hoy día por Política Criminal.

Aunque hoy en día se asumen como tareas indiscutidas de la Política Criminal las relacionadas con la prevención, la definición y la reacción frente al delito, es necesario conocer que no siempre ha sido así, como pone de manifiesto la evolución del concepto de Política Criminal desde su nacimiento en el siglo XVIII.

Conocer los diferentes sentidos en que se habla de Política Criminal hace necesario, más que ofrecer una definición de la misma, comprender distinciones analíticamente relevantes: así, por un lado, la existente entre la Política Criminal como actividad política –la denominada en otras ocasiones política criminal aplicada o política criminal práctica– y la Política Criminal como actividad teórica, también denominada política cri

In [ ]:
print(response['message']['thinking'])

Based on the visual content of the image, I will transcribe the text starting from the left page and moving to the right page, adhering to the natural reading order (top to bottom, left to right).

**Left Page:**
1.  **Header:** "OBJETIVOS DE LA LECCIÓN"
2.  **Paragraph 1:** "En este primer capítulo se trata de acercarse al concepto de Política Criminal y de tener en cuenta los distintos sentidos en que nos podemos referir a la misma para comprender qué se entiende hoy día por Política Criminal."
3.  **Paragraph 2:** "Aunque hoy en día se asumen como tareas indiscutidas de la Política Criminal las relacionadas con la prevención, la definición y la reacción frente al delito, es necesario conocer que no siempre ha sido así, como pone de manifiesto la evolución del concepto de Política Criminal desde su nacimiento en el siglo XVIII."
4.  **Paragraph 3:** "Conocer los diferentes sentidos en que se habla de Política Criminal hace necesario, más que ofrecer una definición de la misma, compre

In [ ]:
!pip install ollama


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [ ]:

from mlx_vlm import load, generate
from mlx_vlm.prompt_utils import apply_chat_template
model_name="mlx-community/Qwen3.5-9B-MLX-4bit"
model_name="mlx-community/Qwen3-VL-4B-Instruct-5bit"
model, processor = load(model_name)


/Users/zzddfge/.pyenv/versions/mlx-31211/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 14 files: 100%|██████████| 14/14 [00:00<00:00, 138818.57it/s]
objc[10117]: Class AVFFrameReceiver is implemented in both /Users/zzddfge/.pyenv/versions/3.12.11/envs/mlx-31211/lib/python3.12/site-packages/av/.dylibs/libavdevice.61.3.100.dylib (0x125c5c3a8) and /Users/zzddfge/.pyenv/versions/3.12.11/envs/mlx-31211/lib/python3.12/site-packages/cv2/.dylibs/libavdevice.61.3.100.dylib (0x3a15e83a8). This may cause spurious casting failures and mysterious crashes. One of the duplicates must be removed or renamed.
objc[10117]: Class AVFAudioReceiver is implemented in both /Users/zzddfge/.pyenv/versions/3.12.11/envs/mlx-31211/lib/python3.12/site-packages/av/.dylibs/libavdevice.61.3.100.dylib (0x125c5c3f8) and /Users

In [ ]:
!pip install --upgrade --force-reinstall --no-cache-dir mlx-vlm


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 694.1/694.1 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.5/527.5 kB 26.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 574.3/574.3 kB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.3/49.3 MB 13.8 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 12.7 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.2/46.2 MB 14.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 16.3 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 15.7 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 616.3/616.3 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.2/34.2 MB 12.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 11.8 MB/s eta 0:00:

In [ ]:
!pip install --upgrade --force-reinstall "mlx-vlm[torch]"
!pip install --upgrade "mistral-common>=1.8.6"

  Using cached mlx_vlm-0.4.0-py3-none-any.whl.metadata (15 kB)
  Using cached mlx-0.31.1-cp312-cp312-macosx_26_0_arm64.whl.metadata (5.9 kB)
  Using cached transformers-5.3.0-py3-none-any.whl.metadata (32 kB)
  Using cached mlx_lm-0.31.1-py3-none-any.whl.metadata (9.5 kB)
  Using cached requests-2.32.5-py3-none-any.whl.metadata (4.9 kB)
  Using cached soundfile-0.13.1-py2.py3-none-macosx_11_0_arm64.whl.metadata (16 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached huggingface_hub-1.7.1-py3-none-any.whl.metadata (13 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached mlx_metal-0.31.1-py3-none-macosx_26_0_arm64.whl.metadata (5.1 kB)
  Using cached sentencepiece-0.2.1-cp312-cp312-macosx_11_0_arm64.whl.metadata (10 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-macosx_11_0_arm64.whl.metadata (7.3 kB)
  Using cached lxml-6.0.2-cp312-cp312-macosx_10_13_univ